# Machine Doctor — Deep Learning Add-on
## Step 4: Train the Neural Network

Real bearing datasets like this one are usually **imbalanced** -- researchers create many fault conditions but only record "Normal" a few times. Without correcting for this, the network could get artificially high accuracy just by leaning toward predicting the majority class. We fix this with **class weighting**: the loss function is told to penalize mistakes on rare classes more heavily.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

data = np.load('/content/drive/MyDrive/machine_doctor_dl/processed_data.npz')
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']
class_names = [str(c) for c in data['class_names']]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| classes:", class_names)

for i, name in enumerate(class_names):
    print(f"  {name}: {np.sum(y_train==i)} train windows ({100*np.mean(y_train==i):.1f}%)")

In [ ]:
class VibrationCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=64, stride=2, padding=32),
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=32, stride=2, padding=16),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=16, stride=2, padding=8),
            nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 32), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = VibrationCNN(num_classes=len(class_names)).to(device)

In [ ]:
def to_loader(X, y, batch_size=64, shuffle=False):
    X_t = torch.tensor(X).unsqueeze(1)
    y_t = torch.tensor(y)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, shuffle=True)
test_loader = to_loader(X_test, y_test, shuffle=False)

class_counts = np.array([np.sum(y_train == i) for i in range(len(class_names))])
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * len(class_names)
class_weights = class_weights.to(device)
print("Class weights:", dict(zip(class_names, class_weights.cpu().numpy().round(2))))

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss_sum += criterion(outputs, y_batch).item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return loss_sum / total, correct / total

EPOCHS = 20
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
best_test_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    correct, total, loss_sum = 0, 0, 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * X_batch.size(0)
        correct += (outputs.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.size(0)

    train_loss, train_acc = loss_sum / total, correct / total
    test_loss, test_acc = evaluate(model, test_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.3f} | test_loss={test_loss:.4f} test_acc={test_acc:.3f}")

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/machine_doctor_dl/best_model.pt')

print(f"\nBest test accuracy: {best_test_acc:.3f} (checkpoint saved to Drive)")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["test_loss"], label="test")
axes[0].set_title("Loss over epochs")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["test_acc"], label="test")
axes[1].set_title("Accuracy over epochs")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/machine_doctor_dl/training_curves.png', dpi=100)
plt.show()

print("\nWhat to look for: train and test lines staying close together = good sign.")
print("Test accuracy going UP while test LOSS also starts going up = overfitting warning.")